# **ST 554 Final Project**
---
Authored by: Jamie Loring

Collaborator: Dr. Justin Post (code from lecture videos & slides)

## **Setup**

The code below imports the required modules that are needed for this project.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, OneHotEncoder, PCA, \
                               StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

#from sklearn import linear_model
#from math import sqrt
#from sklearn.model_selection import train_test_split, cross_validate
#from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, accuracy_score
#from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge, ElasticNet, \
                                 #LogisticRegressionCV, LassoCV, RidgeCV, ElasticNetCV
#from sklearn.preprocessing import PolynomialFeatures

#from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
#from sklearn.model_selection import GridSearchCV
#from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
#from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier


#from pyspark.sql.functions import when

The code below sets up our `spark` object and only allows errors to print out in the future (i.e., it suppresses warnings).

In [2]:
spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 09:44:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


The code below reads in the `power_ml_data.csv` file from the provided URL using `pandas`.

In [3]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


Now, we convert our `pandas dataframe` to a `spark` SQL style dataframe. This new object will be called `power_ML`. For better readability, throughout this project, I will display results using `.toPandas()` and `.head()` rather than `.show()`. It is important to note that this choice does not change `power_ML` back to a `pandas` or `pandas-on-spark` dataframe; rather, it just changes the way it is *displayed*.

In [5]:
power_ML = spark.createDataFrame(power_data)
power_ML.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


We are going to treat the `Power_Zone_3` variable as our response variable. We want to imagine that we know that the `Power_Zone_3` reading is going to go offline in the future, and we need to be able to predict that value appropriately using the other variables in the dataset.

## **Fitting the Model**

We are going to fit an elastic net model using CV without a training & testing split. Before we can fit our model, we have to perform the required transformations using `MLlib` functions. All (nested) transformations will be saved as objects so 1) We can display our progress through the transformations and 2) We can check to make sure we do not miss any transformations in the final pipeline.

First, we need to check to see how the `Hour` column is stored. 

In [6]:
power_ML.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

Now that we have confirmed that it's not a `DoubleType`, we will need to use an SQL transformer to cast the `Hour` variable as a `DoubleType`.

In [7]:
cast_hour = SQLTransformer(
    statement = """
                SELECT *, CAST(Hour as DOUBLE) as Hour_Double_Type FROM __THIS__
                """
            )

tf_1 = cast_hour.transform(power_ML)
tf_1.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0


Next, we need to binarize the updated `Hour_Double_Type` column based on it being less than 6.5 or not. We are essentially creating a night vs. day variable.

In [8]:
binarize_hour = Binarizer(threshold = 6.5, inputCol = "Hour_Double_Type", outputCol = "Night_vs_Day")

tf_2 = binarize_hour.transform(tf_1)
tf_2.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0


Continuing on, we need to one-hot encode the `Month` variable. Since `Month` is already an integer, we do _**not**_ need to use `StringIndexer()`.

In [10]:
encoder_OHE = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_OHE"])
model_OHE = encoder_OHE.fit(power_ML)

tf_3 = model_OHE.transform(tf_2)
tf_3.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Our next step is to run a PCA fit on the following variables:
- `Temperature`
- `Humidity`
- `Wind_Speed`
- `General_Diffuse_Flows`
- `Diffuse_Flows`

We will also need to standarize our predictors using `StandardScaler()`.

**Note:** This will be a multi-step process, outlined with in-line comments in the code chunk below.

In [12]:
# use VectorAssembler() to place the above variables in a column together for use with the PCA() estimator
assembler_PCA = VectorAssembler(
                inputCols = ["Temperature", "Humidity", "Wind_Speed",
                             "General_Diffuse_Flows", "Diffuse_Flows"],
                outputCol = "features_for_PCA",
                handleInvalid = "keep"
             )

# update transformations
tf_4 = assembler_PCA.transform(tf_3)

# use StandardScaler() to standardize predictors
scaler_PCA = StandardScaler(inputCol = "features_for_PCA", outputCol = "features_for_PCA_scaled",
                            withStd = True, withMean = True)
# fit scaler_PCA
scaler_fit = scaler_PCA.fit(tf_4)

# update transformations
tf_5 = scaler_fit.transform(tf_4)

# run PCA, using 2 PC's in the transformation
PCA_func = PCA(k = 2, inputCol = "features_for_PCA_scaled", outputCol = "PCA_features_scaled")

# fit the PCA model
PCA_model = PCA_func.fit(tf_5)

# update transformations
tf_6 = PCA_model.transform(tf_5)

# show updated transformations
tf_6.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463714, 0.8078678829472261]"
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.1029210638806535, 0.815247622280639]"
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.112066263379101, 0.8227151924829956]"
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.1435031847422192, 0.8329135817940962]"
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.1824440036626997, 0.8444681624812326]"


We now need to put our model predictors into a single column called features. We can do this with `VectorAssembler()`. We will also use `StandardScaler()` to standardize our features. Since `PCA_features_scaled` has already been standardized, we do not need to re-standardize it, so this will be a multi-step process.

In [13]:
# assemble features vector without PCA for scaling
assembler_features = VectorAssembler(
                          inputCols = ["Night_vs_Day", "Power_Zone_1",
                                       "Power_Zone_2", "Month_OHE"],
                          outputCol = "features_pre_scale",
                          handleInvalid = "keep"
                    )

# update transformations
tf_7 = assembler_features.transform(tf_6)

# use StandardScaler() to standardize features -- remember PCA_features_scaled is already standardized!
scaler_features = StandardScaler(inputCol = "features_pre_scale", outputCol = "features_scaled",
                                 withStd = True, withMean = True)

# fit scaler_features
scaler_fit_feat = scaler_features.fit(tf_7)

# update transformations
tf_8 = scaler_fit_feat.transform(tf_7)

# use VectorAssembler() again to combine PCA_features_scaled and features_scaled into one features column
assembler_features_final = VectorAssembler(
                                inputCols = ["PCA_features_scaled", "features_scaled"],
                                outputCol = "features",
                                handleInvalid = "keep"
                        )

# update transformations
tf_9 = assembler_features_final.transform(tf_8)

# show updated transformations
tf_9.toPandas().head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour,Hour_Double_Type,Night_vs_Day,Month_OHE,features_for_PCA,features_for_PCA_scaled,PCA_features_scaled,features_pre_scale,features_scaled,features
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.559, 73.8, 0.083, 0.051, 0.119]","[-2.1079477438809433, 0.3542085264292604, -0.7...","[2.0690743213463714, 0.8078678829472261]","(0.0, 34055.6962, 16128.87538, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, 0.24130775579578978, -0....","[2.0690743213463714, 0.8078678829472261, -1.55..."
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.414, 74.5, 0.083, 0.07, 0.085]","[-2.1328903699941857, 0.3991947174962608, -0.7...","[2.1029210638806535, 0.815247622280639]","(0.0, 29814.68354, 19375.07599, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.35350356900601565, -0...","[2.1029210638806535, 0.815247622280639, -1.554..."
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.313, 74.5, 0.08, 0.062, 0.1]","[-2.1502641992178924, 0.3991947174962608, -0.8...","[2.112066263379101, 0.8227151924829956]","(0.0, 29128.10127, 19006.68693, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.449798237837335, -0.3...","[2.112066263379101, 0.8227151924829956, -1.554..."
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[6.121, 75.0, 0.083, 0.091, 0.096]","[-2.183291676554047, 0.4313277111155467, -0.79...","[2.1435031847422192, 0.8329135817940962]","(0.0, 28228.86076, 18361.09422, 0.0, 1.0, 0.0,...","[-1.5544690531019132, -0.5759186911227896, -0....","[2.1435031847422192, 0.8329135817940962, -1.55..."
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0,0.0,0.0,"(0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[5.921, 75.7, 0.081, 0.048, 0.085]","[-2.217695298779209, 0.47631390218254716, -0.8...","[2.1824440036626997, 0.8444681624812326]","(0.0, 27335.6962, 17872.34043, 0.0, 1.0, 0.0, ...","[-1.5544690531019132, -0.7011869790980542, -0....","[2.1824440036626997, 0.8444681624812326, -1.55..."


Finally, we rename the response variable, `Power_Zone_3`, as `label`. We will also use the SQL transformer to select the `features` vector that was created in the previous step.

In [14]:
label_and_features = SQLTransformer(
    statement = """
                SELECT features, Power_Zone_3 as label FROM __THIS__
                """
            )

# update transformations
tf_10 = label_and_features.transform(tf_9)
tf_10.toPandas().head()

,features,label
0,"[2.0690743213463714, 0.8078678829472261, -1.55...",20240.96386
1,"[2.1029210638806535, 0.815247622280639, -1.554...",20131.08434
2,"[2.112066263379101, 0.8227151924829956, -1.554...",19668.43373
3,"[2.1435031847422192, 0.8329135817940962, -1.55...",18899.27711
4,"[2.1824440036626997, 0.8444681624812326, -1.55...",18442.40964


We can now put all of our transformations into the pipeline. We will have 10 of them. They are:
- `cast_hour`
- `binarize_hour`
- `encoder_OHE`
- `assembler_PCA`
- `scaler_PCA`
- `PCA_func`
- `assembler_features`
- `scaler_features`
- `assembler_features_final`
- `label_and_features`
- `lr`, which will be the instance of our `LinearRegression()`

**Note:** Only the transformers and estimators need to be put into the pipeline since the pipeline will automatically handle which ones need to be fit. The fit objects up to this point were only created to show the progress of the transformations and check their functionality, but they themselves do not go into the pipeline!

*Example:* `encoder_OHE` *goes into the pipeline rather than* `model_OHE`

In [15]:
lr = LinearRegression()
pipeline = Pipeline(stages = [cast_hour, binarize_hour, encoder_OHE, assembler_PCA,
                              scaler_PCA, PCA_func, assembler_features, scaler_features,
                              assembler_features_final, label_and_features, lr])

We are now ready to fit our elastic net model. We are provided with the following grid for `regParam` and `elasticNetParam`:
- `regParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- `elasticNetParam`: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

Since our `pipeline` object is already set up, we have the following remaining steps to complete in creating our model:
- build the parameter grid
- use cross-validation (5 folds) to choose the best combination of parameters using RMSE
- fit the model

**Note:** The code below will take about 12 minutes to run, please be patient!

In [16]:
# build parameter grid
paramGrid = ParamGridBuilder() \
            .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
            .build()


# set up cross validation with pipeline
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = 'rmse'),
                          numFolds = 5,
                          seed = 44)

# fit the model using cross-validation to choose the best model
cvModel = crossval.fit(power_ML)

Although the `.bestModel` attribute will store the best model as a result of cross-validation by default, we want to look to see what model is the best. The code below prints the RMSE for the model fit with its corresponding parameters, `regParam` and `elasticNetParam`. I then sort this output in ascending order so we can see the lowest RMSE first (along with its corresponding parameters).

In [17]:
my_list = []

for i in range(len(paramGrid)):
    my_list.append([cvModel.avgMetrics[i], paramGrid[i].values()])
    
my_list.sort(key=lambda x: x[0]) #specifies sorting by avgMetrics rather than the dict_values
my_list

[[2175.025614452874, dict_values([0.25, 0.5])],
 [2175.033518764532, dict_values([0.5, 0.25])],
 [2175.0335454632104, dict_values([0.5, 0.05])],
 [2175.0341016574057, dict_values([0.5, 0.1])],
 [2175.0349079585444, dict_values([0.05, 0.5])],
 [2175.035462853383, dict_values([0.25, 0.1])],
 [2175.035512165499, dict_values([0.1, 0.95])],
 [2175.035587292835, dict_values([0.05, 0.98])],
 [2175.0356041252753, dict_values([0.05, 0.9])],
 [2175.0356236594253, dict_values([0.05, 0.95])],
 [2175.035731007239, dict_values([0.05, 0.99])],
 [2175.0357346658175, dict_values([0.05, 1.0])],
 [2175.035803359566, dict_values([0.05, 0.25])],
 [2175.0358332557826, dict_values([0.1, 0.0])],
 [2175.035841619817, dict_values([0.1, 0.1])],
 [2175.0358417832153, dict_values([0.25, 0.0])],
 [2175.035842577645, dict_values([0.05, 0.75])],
 [2175.0358509369303, dict_values([0.05, 0.0])],
 [2175.035880590475, dict_values([0.0, 0.25])],
 [2175.0358805904766, dict_values([0.0, 1.0])],
 [2175.035880590478, dict_val

Thus, the best multiple linear regression model is one with a regularization parameter of 0.25 and an elastic net parameter of 0.5. This tells us that we have a combination of L1 and L2 penalties, hence, we are fitting an elastic net model! We can now print the intercept and coefficients of this best model.

In [19]:
# need to use indexing since the model is the last stage of the pipeline
#FIX COEFFICIENTS
print('Intercept:', cvModel.bestModel.stages[-1].intercept)
print('sqft, multi_story_ind, year_built, beds, baths_full coeffs:', cvModel.bestModel.stages[-1].coefficients)
print('Best RMSE on Training (Full) Data:', round(my_list[0][0], 5))

Intercept: 17831.197607816746
sqft, multi_story_ind, year_built, beds, baths_full coeffs: [281.3375940841656,-508.54885234099834,-933.3422279345826,4380.973673101204,582.7712870595021,0.0,1670.643619500629,1521.0143011644395,1488.0485558207417,1932.6899471239326,1411.984181133519,1784.2129277382987,3556.71978908298,2402.8363020388033,373.5950605539571,-54.722433929873375,504.5217858788961]
Best RMSE on Training (Full) Data: 2175.02561


## **Streaming**

### **Reading a Stream**

### **Transformations and Aggregations**

### **Writing Step**

## **Producing the Data**